# Train locally

Use the project `.venv` kernel. Run `data_prep.ipynb` first.
Set `RUN_NAME` and the settings below. `SMOKE_RUN=True` runs a short check
with full-fold validation. Use a new run name for changed settings.
Paths and W&B keys come from `.env` or the environment.
See [training settings](../../src/birdclef/models/encoders/efficientnet.md).

In [ ]:
import json
import os

import torch

from birdclef.config import RuntimeConfig, TrainConfig, get_config
from birdclef.models import AudioExperimentConfig, AudioModelConfig

RUN_NAME = "audio-cnn/partial-unfreeze"
SMOKE_RUN = False
WANDB_OFFLINE = False  # True to log locally and sync later with wandb sync.
CPU_THREADS = 4

torch.set_num_threads(CPU_THREADS)

config = get_config(
    AudioExperimentConfig,
    fold=0,
    smoke=SMOKE_RUN,
    model=AudioModelConfig(
        pretrained=True,
        encoder_lr=3e-5,
        head_lr=3e-4,
        unfreeze_epoch=1,
        unfreeze_last_n=2,  # None = all encoder
    ),
    tracking={"offline": WANDB_OFFLINE},
    runtime=RuntimeConfig(accelerator="auto", devices=1),
    loader={
        "batch_size": 4 if SMOKE_RUN else 32,
        "val_batch_size": 16,
        "predict_batch_size": 16,
        "num_workers": 0,
        "seed": 42,
    },
    train=TrainConfig(
        max_epochs=1 if SMOKE_RUN else 6,
        checkpoint_metric="val/auc",
        checkpoint_mode="max",
        early_stopping_patience=3,
        log_every_n_steps=1,
    ),
)
RUN_DIR = config.data.output_dir / config.run_name(RUN_NAME)
print(f"Run directory: {RUN_DIR}")

                            AudioExperimentConfig                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting                       ┃ Value                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ app_name                      │ birdclef                                  │
│ logging.directory             │ C:\_GitHub\birdclef-2026\logs             │
│ logging.level                 │ INFO                                      │
│ logging.formatter             │ ColoredFormatter                          │
│ logging.date_format           │ %d-%m-%Y %H:%M:%S                         │
│ logging.full_color            │ True                                      │
│ logging.include_function      │ True                                      │
│ logging.success_level         │ 69                                        │
├───────────────────────────────┼───────────────────────────────────────────┤
│ data.competition_dir          │ C:\_GitHub\birdclef-2026\data\competition │
│ data.output_dir               │ C:\_GitHub\birdclef-2026\outputs          │
├───────────────────────────────┼───────────────────────────────────────────┤
│ split.validation_fraction     │ 0.2                                       │
│ split.seed                    │ 42                                        │
│ split.group_column            │ None                                      │
│ split.stratify                │ False                                     │
├───────────────────────────────┼───────────────────────────────────────────┤
│ tracking.project              │ birdclef-2026                             │
│ tracking.entity               │ dmykhailov-kyiv-school-of-economics       │
│ tracking.offline              │ False                                     │
│ tracking.api_key              │ **********                                │
├───────────────────────────────┼───────────────────────────────────────────┤
│ loader.batch_size             │ 32                                        │
│ loader.val_batch_size         │ 16                                        │
│ loader.predict_batch_size     │ 16                                        │
│ loader.num_workers            │ 0                                         │
│ loader.pin_memory             │ False                                     │
│ loader.seed                   │ 42                                        │
│ loader.soundscape_probability │ 0.1                                       │
├───────────────────────────────┼───────────────────────────────────────────┤
│ lr_scheduler.name             │ none                                      │
│ lr_scheduler.t_max            │ 10                                        │
│ lr_scheduler.min_lr           │ 0.0                                       │
│ lr_scheduler.factor           │ 0.1                                       │
│ lr_scheduler.patience         │ 2                                         │
│ lr_scheduler.monitor          │ val/loss                                  │
│ lr_scheduler.mode             │ min                                       │
├───────────────────────────────┼───────────────────────────────────────────┤
│ train.max_epochs              │ 10                                        │
│ train.seed                    │ 42                                        │
│ train.checkpoint_metric       │ val/auc                                   │
│ train.checkpoint_mode         │ max                                       │
│ train.checkpoint_top_k        │ 1                                         │
│ train.early_stopping_patience │ 5                                         │
│ train.accumulate_grad_batches │ 1                                         │
│ train.gradient_clip_val       │ 1.0                                       │
│ train.log_every_n_steps       │ 1                                         │
├─────────────────────

Run directory: C:\_GitHub\birdclef-2026\outputs\audio-cnn\gradual-unfreeze


## Train

Use one device with `%run`. Logs and progress show here; interrupt the cell to stop.
Run the same name to continue. Raise `max_epochs` for more training.
Use a new name for changed data or settings. API keys are not saved in the config.

In [2]:
if config.tracking.api_key is not None:
    os.environ["BCLEF_TRACKING__API_KEY"] = config.tracking.api_key.get_secret_value()

config_path = config.data.output_dir / "configs" / f"{config.run_name(RUN_NAME)}.json"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")


1604

In [3]:
%run -m birdclef.models.fit -- --config "{config_path}" --name "{RUN_NAME}"

W1027 20:06:09.296000 17116 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


27-10-2026 20:06:10  INFO      [models/audio_cnn/fit:66]  Run: audio-cnn/gradual-unfreeze; training examples: 35769; validation windows: 149


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: C:\Users\dennm\_netrc
wandb: Currently logged in as: dmykhailov (kaggle-casmi) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


Seed set to 42
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 5050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not b

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ criterion │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 1 │ encoder   │ Sequential        │  4.0 M │ eval  │     0 │
│ 2 │ pool      │ AdaptiveAvgPool2d │      0 │ train │     0 │
│ 3 │ head      │ Sequential        │  299 K │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

...

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


epoch,▁▁▁▁▁▂▂▃▃▃▄▄▄▄▄▄▄▄▄▅▆▆▆▆▆▆▆▇▇▇▇▇▇███████
lr-AdamW/encoder,▁▁▁▁▁▁▁▁▁
lr-AdamW/head,▁▁▁▁▁▁▁▁▁
train/loss_epoch,█▄▆▃▂▂▁▁
train/loss_step,█▆▆▆▄▄▄▆▇▄▅▇▄▄▅▄▃▃▄▃▂▂▃▃▃▁▃▂▂▃▂▂▁▁▂▃▂▃▁▃
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇██
val/auc,▁▃▆▇▇███
val/auc_classes,▁▁▁▁▁▁▁▁
val/loss,▃▂▁▃▄▆▇█
epoch,8
lr-AdamW/encoder,3e-05


SystemExit: 1

c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\IPython\core\interactiveshell.py:2940: UserWarning: Unknown failure executing module: <birdclef.models.audio_cnn.fit>
  warn('Unknown failure executing module: <%s>' % mod_name)


## Results

These reports use the best checkpoint and the full saved fold. Classes with no
positives are skipped in macro AUC. A smoke score only checks that the code works.
This is one fold, not an OOF score.

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_json(RUN_DIR / "validation-metrics.json", typ="series"))
display(
    pd.read_csv(RUN_DIR / "validation-class-metrics.csv")
    .sort_values("auc", ascending=False)
    .head(15)
)

## Optional CPU check

Load the saved model and predict one real minute (12 windows). Timing includes
checkpoint loading, audio reading, Log-Mel and prediction. The CSV checks format;
it is not a test submission or a full runtime check.

In [ ]:
import time

from birdclef.competition import ROW_ID, soundscape_windows
from birdclef.models import AudioClassifier
from birdclef.submission import build_submission, write_submission

cpu_config = config.model_copy(deep=True)
cpu_config.runtime = RuntimeConfig(accelerator="cpu", devices=1, precision="32-true")
restored = AudioClassifier.from_run(RUN_DIR, config=cpu_config)
classes = restored.classes
all_windows = soundscape_windows(config.data.competition_dir / "train_soundscapes")
validation_ids = pd.read_csv(RUN_DIR / "validation-predictions.csv", usecols=[ROW_ID])[ROW_ID]
filename = all_windows.loc[all_windows[ROW_ID].isin(validation_ids), "filename"].iloc[0]
minute = all_windows.loc[all_windows.filename.eq(filename)].reset_index(drop=True)
assert len(minute) == 12
started = time.perf_counter()
probabilities = restored.predict(minute)
elapsed = time.perf_counter() - started

template = pd.DataFrame(0.0, index=range(len(minute)), columns=classes)
template.insert(0, ROW_ID, minute[ROW_ID].to_numpy())
submission = build_submission(probabilities, template)
path = write_submission(submission, RUN_DIR / "minute-sanity-submission.csv", template)
benchmark = {
    "seconds": elapsed,
    "audio_seconds": 60,
    "windows": len(minute),
    "cpu_threads": CPU_THREADS,
    "torch_version": torch.__version__,
    "includes": "checkpoint load, Trainer setup, decode, Log-Mel, CNN",
}
(RUN_DIR / "cpu-benchmark.json").write_text(json.dumps(benchmark, indent=2), encoding="utf-8")
display(pd.Series(benchmark))
display(submission.head())
print(f"Sanity-check CSV: {path}")
